# Set up

In [13]:
import pandas as pd
import os
import matplotlib as plt

from surprise import Reader, AlgoBase, Dataset, accuracy
from surprise.model_selection import train_test_split
import numpy as np
from collections import defaultdict
import recmetrics
from sklearn.metrics import ndcg_score

from sklearn.preprocessing import StandardScaler

from surprise.model_selection import cross_validate, KFold

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.preprocessing import normalize

In [14]:
SEED = 42; TOP_K = 10; RELEVANCE_THRESHOLD = 7; MIN_ITEM_RATINGS = 500
N_USERS = 5000; N_USERS_COVERAGE = 300

np.random.seed(SEED)
ratings_df = pd.read_csv("../processed_data/explicit_ratings.csv")

# We shuffle the users
all_users = ratings_df["user_id"].unique()
np.random.shuffle(all_users)

# Readers for our ranking (implemented wide as some ratings go above 10. Showed improvement!)
reader      = Reader(rating_scale=(1, 10))
reader_wide = Reader(rating_scale=(1, 20))

# We sample the 5000 users
sample = ratings_df[ratings_df["user_id"].isin(all_users[:N_USERS])][["user_id", "anime_id", "rating"]]
data = Dataset.load_from_df(sample, reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=SEED) # Normal 80/20 split

# For coverage, we are using only 300 people as the matrix with 6000 catalog items would be 1.8M predictions and not 30M with 5k users.
sample_cov = ratings_df[ratings_df["user_id"].isin(all_users[:N_USERS_COVERAGE])][["user_id", "anime_id", "rating"]]
data_cov = Dataset.load_from_df(sample_cov, reader_wide)
trainset_cov = data_cov.build_full_trainset() 
# We are using the whole dataset for training as we want to have as much information for computing coverage
antitestset_cov = trainset_cov.build_anti_testset() # all user item pairs which are not in the trainset
catalog = [trainset_cov.to_raw_iid(iid) for iid in trainset_cov.all_items()]

genre_cols = ['Action', 'Adventure', 'Cars', 'Comedy', 'Dementia', 'Demons', 'Drama', 'Ecchi', 'Fantasy', 'Game', 'Harem', 'Historical', 'Horror', 'Josei', 'Kids', 'Magic', 'Martial Arts', 'Mecha', 'Military', 'Music', 'Mystery', 'Parody', 'Police', 'Psychological', 'Romance', 'Samurai', 'School', 'Sci-Fi', 'Seinen', 'Shoujo', 'Shoujo Ai', 'Shounen', 'Shounen Ai', 'Slice of Life', 'Space', 'Sports', 'Super Power', 'Supernatural', 'Thriller', 'Vampire', 'Yaoi']
anime_meta = pd.read_csv("../processed_data/anime_processed.csv")[["MAL_ID"] + genre_cols].set_index("MAL_ID")

In [15]:
import gc

def get_top_n(predictions, n=TOP_K):
    """
        Function takes prediction and sorts them to return the top K recommendations.
        - Sorts predictions based on estimated rating (in a descending manner)
        - Limits to only top K numbers
    """
    user_preds = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_preds[uid].append((iid, est, true_r))
    return {uid: sorted(v, key=lambda x: x[1], reverse=True)[:n] for uid, v in user_preds.items()}

def eval_accuracy(predictions, n=TOP_K, threshold=RELEVANCE_THRESHOLD):
    """
        Function which computes all relevant regression, classification and ranking metrics.
        Responsible for MAE, RMSE, Precision, Recall, and NDCG
    """
    rmse = accuracy.rmse(predictions, verbose=False)
    mae  = accuracy.mae(predictions, verbose=False)
    user_preds = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_preds[uid].append((iid, est, true_r))
    precisions, recalls, ndcgs = [], [], []
    for uid, preds in user_preds.items():
        top = sorted(preds, key=lambda x: x[1], reverse=True)[:n]
        hits      = sum(r >= threshold for _, _, r in top)
        total_rel = sum(r >= threshold for _, _, r in preds)
        precisions.append(hits / n)
        recalls.append(hits / total_rel if total_rel > 0 else 0)
        true_r = [r for _, _, r in top]; est_r = [e for _, e, _ in top]
        if len(true_r) > 1:
            ndcgs.append(ndcg_score([true_r], [est_r]))
    return {"RMSE": round(rmse, 4), "MAE": round(mae, 4),
            f"Precision@{n}": round(np.mean(precisions), 4),
            f"Recall@{n}":    round(np.mean(recalls), 4),
            f"NDCG@{n}":      round(np.mean(ndcgs), 4)}

def eval_coverage(predictions, catalog, feature_df, n=TOP_K):
    """
        Function which computes not just accuracy metrics. 
        Due to the long-tail effect, the full dataset is used for training and the antitest is also utilized.
    """
    top_n     = get_top_n(predictions, n)
    full_recs = [[iid for iid, _, _ in v] for v in top_n.values() if len(v) == n]
    cov  = recmetrics.prediction_coverage(full_recs, catalog) if full_recs else 0.0
    pers = recmetrics.personalization(full_recs)              if full_recs else 0.0
    ils  = recmetrics.intra_list_similarity(full_recs, feature_df)
    return {"Coverage": round(cov, 4), "Personalization": round(pers, 4),
            "Intra-List Similarity": round(ils, 4)}

# Non-Personalized Recommenders

## Popular Recommender

In [ ]:
class PopularRecommender(AlgoBase):
    def __init__(self, min_ratings=MIN_ITEM_RATINGS):
        AlgoBase.__init__(self)
        self.min_ratings = min_ratings

    def fit(self, trainset):
        """
            This implemented fit functions for the popular recommender will:
            - aggregate the counts of ratings for each anime
            - will discard any anime that has a lower amount of ratings than the defined threshold
            - returns the filtered options
        """
        AlgoBase.fit(self, trainset)
        df = pd.DataFrame([(i, r) for (_, i, r) in trainset.all_ratings()], columns=["item", "rating"])
        stats = df.groupby("item").agg(count=("rating", "count"), mean=("rating", "mean")) # aggregating part
        self.item_means = stats.loc[stats["count"] >= self.min_ratings, "mean"] # filtering part
        self.global_mean = trainset.global_mean
        return self

    def estimate(self, u, i):
        if i in self.item_means.index:
            return self.item_means[i]
        return self.global_mean

In [5]:
popular = PopularRecommender()

popular.fit(trainset)
acc = eval_accuracy(popular.test(testset))

popular.fit(trainset_cov)
cov_preds = popular.test(antitestset_cov)
cov = eval_coverage(cov_preds, catalog, anime_meta)
del cov_preds; gc.collect()

pd.Series({**acc, **cov})

RMSE                     1.6907
MAE                      1.2893
Precision@10             0.8003
Recall@10                0.5136
NDCG@10                  0.9728
Coverage                 0.6500
Personalization          0.1593
Intra-List Similarity    0.3046
dtype: float64

## Random Recommender

In [6]:
class RandomRecommender(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        ratings = [r for (_, _, r) in trainset.all_ratings()]
        self.mean = np.mean(ratings)
        self.std  = np.std(ratings)
        return self

    def estimate(self, u, i):
        return np.random.normal(self.mean, self.std)

In [7]:
random_rec = RandomRecommender()

random_rec.fit(trainset)
acc = eval_accuracy(random_rec.test(testset))

random_rec.fit(trainset_cov)
cov_preds = random_rec.test(antitestset_cov)
cov = eval_coverage(cov_preds, catalog, anime_meta)
del cov_preds; gc.collect()

pd.Series({**acc, **cov})

RMSE                      2.4038
MAE                       1.9041
Precision@10              0.7183
Recall@10                 0.4844
NDCG@10                   0.9472
Coverage                 38.9800
Personalization           0.9984
Intra-List Similarity     0.1925
dtype: float64

# Content-Based Recommenders

## Metadata-Based Recommender

In [ ]:
GENRE_COLS = ['Action','Adventure','Cars','Comedy','Dementia','Demons','Drama','Ecchi',
              'Fantasy','Game','Harem','Historical','Horror','Josei','Kids','Magic',
              'Martial Arts','Mecha','Military','Music','Mystery','Parody','Police',
              'Psychological','Romance','Samurai','School','Sci-Fi','Seinen','Shoujo',
              'Shoujo Ai','Shounen','Shounen Ai','Slice of Life','Space','Sports',
              'Super Power','Supernatural','Thriller','Vampire','Yaoi']

_anime_full = pd.read_csv("../processed_data/anime_processed.csv").set_index("MAL_ID")

# We get the top 50 most popular anime studios
_top_studios = _anime_full["Studios"].value_counts()
_top_studios = _top_studios[_top_studios > 50].index

def parse_duration(s):
    import re
    if pd.isna(s):
        return np.nan
    hr = re.search(r'(\d+)\s*hr', s)
    mn = re.search(r'(\d+)\s*min', s)
    return (int(hr.group(1)) * 60 if hr else 0) + (int(mn.group(1)) if mn else 0)

def build_features(anime_df):
    genre_feats = anime_df[GENRE_COLS].fillna(0) # fill empty genres with 0

    dur_raw = anime_df["Duration"].apply(parse_duration)
    type_medians = anime_df.groupby("Type")["Duration"].apply(
        lambda x: x.apply(parse_duration).median()
    )
    dur_vals = dur_raw.fillna(anime_df["Type"].map(type_medians))

    ep_vals = anime_df[["Episodes"]].fillna(anime_df["Episodes"].median()) # imputation with median for episode count (more robust as some animes have 1000s of episodes)
    num_df  = pd.DataFrame({"Episodes": ep_vals["Episodes"], "Duration": dur_vals}, index=anime_df.index)
    num_df  = num_df.fillna(num_df.median())

    # Scaling the numerical features
    sc = StandardScaler()
    num_feats = pd.DataFrame(sc.fit_transform(num_df), index=anime_df.index, columns=num_df.columns)

    # Input relevant studio names to the dataframe
    studio_col = anime_df["Studios"].apply(lambda x: x if x in _top_studios else "Other")
    cat_df = pd.DataFrame({
        "Type":   anime_df["Type"],
        "Rating": anime_df["Rating"],
        "Source": anime_df["Source_grouped"].fillna("Unknown"),
        "Studio": studio_col,
    }, index=anime_df.index)
    cat_feats = pd.get_dummies(cat_df)

    feats = pd.concat([genre_feats, num_feats, cat_feats], axis=1).fillna(0)
    return feats.values.astype(float)

In [6]:
class MetadataRecommender(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)

        raw_ids   = [trainset.to_raw_iid(i) for i in trainset.all_items()]
        anime_sub = _anime_full.loc[[r for r in raw_ids if r in _anime_full.index]]

        feat_matrix = build_features(anime_sub)
        norms = np.linalg.norm(feat_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1
        self.feat_matrix = feat_matrix / norms
        self.item_idx    = {raw_id: idx for idx, raw_id in enumerate(anime_sub.index)}

        self.user_items = {
            u: [(self.item_idx[trainset.to_raw_iid(j)], r)
                for j, r in trainset.ur[u]
                if trainset.to_raw_iid(j) in self.item_idx]
            for u in trainset.all_users()
        }
        return self

    def test(self, testset, verbose=False):
        from collections import defaultdict
        from surprise import Prediction

        user_tests = defaultdict(list)
        for uid, iid, r_ui in testset:
            user_tests[uid].append((iid, r_ui))

        predictions = []
        global_mean = self.trainset.global_mean
        lo, hi = self.trainset.rating_scale

        for raw_uid, test_items in user_tests.items():
            try:
                iuid     = self.trainset.to_inner_uid(raw_uid)
                user_rated = self.user_items.get(iuid, [])
            except ValueError:
                user_rated = []

            known   = [(iid, r) for iid, r in test_items if iid in self.item_idx and user_rated]
            unknown = [(iid, r) for iid, r in test_items if iid not in self.item_idx or not user_rated]

            if known:
                test_idxs  = [self.item_idx[iid] for iid, _ in known]
                rated_idxs = [idx for idx, _ in user_rated]
                ratings    = np.array([r for _, r in user_rated])

                result = self.feat_matrix[test_idxs].dot(self.feat_matrix[rated_idxs].T)
                sims   = result.toarray() if hasattr(result, "toarray") else np.asarray(result)

                for j, (raw_iid, r_ui) in enumerate(known):
                    s   = sims[j]
                    pos = s > 0
                    est = float(s[pos] @ ratings[pos] / s[pos].sum()) if pos.any() else global_mean
                    predictions.append(Prediction(raw_uid, raw_iid, r_ui, min(hi, max(lo, est)), {}))

            for raw_iid, r_ui in unknown:
                predictions.append(Prediction(raw_uid, raw_iid, r_ui, global_mean, {}))

        return predictions

    def estimate(self, u, i):
        if str(i).startswith("UKN__"):
            return self.trainset.global_mean
        raw_i = self.trainset.to_raw_iid(i)
        if raw_i not in self.item_idx or not self.user_items.get(u):
            return self.trainset.global_mean
        vec_i      = self.feat_matrix[self.item_idx[raw_i]]
        rated_idxs = [idx for idx, _ in self.user_items[u]]
        ratings    = np.array([r for _, r in self.user_items[u]])
        sims       = self.feat_matrix[rated_idxs] @ vec_i
        pos        = sims > 0
        if not pos.any():
            return self.trainset.global_mean
        return float(sims[pos] @ ratings[pos] / sims[pos].sum())

In [7]:
meta_rec = MetadataRecommender()

meta_rec.fit(trainset)
acc = eval_accuracy(meta_rec.test(testset))

meta_rec.fit(trainset_cov)
cov_preds = meta_rec.test(antitestset_cov)
cov = eval_coverage(cov_preds, catalog, anime_meta)
del cov_preds; gc.collect()

pd.Series({**acc, **cov})

RMSE                      1.4649
MAE                       1.1096
Precision@10              0.7774
Recall@10                 0.5070
NDCG@10                   0.9608
Coverage                 10.7300
Personalization           0.9454
Intra-List Similarity     0.3806
dtype: float64

In [ ]:
cv_results = cross_validate(
    MetadataRecommender(), data,
    measures=["RMSE", "MAE"],
    cv=KFold(n_splits=5, random_state=SEED),
    return_train_measures=True,
    n_jobs=1,
    verbose=False
)
pd.DataFrame(cv_results)[["test_rmse", "test_mae", "train_rmse", "train_mae"]].agg(["mean", "std"])

## BoW Recommender

In [16]:
_synopsis_df = pd.read_csv("../anime_with_synopsis.csv")[["MAL_ID", "sypnopsis"]].set_index("MAL_ID")
_synopsis_df["sypnopsis"] = _synopsis_df["sypnopsis"].fillna("")

In [17]:
class BoWRecommender(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)

        raw_ids   = [trainset.to_raw_iid(i) for i in trainset.all_items()]
        known     = [r for r in raw_ids if r in _synopsis_df.index]
        anime_sub = _synopsis_df.loc[known]

        vectorizer       = CountVectorizer(stop_words="english", max_features=5000)
        bow_matrix       = vectorizer.fit_transform(anime_sub["sypnopsis"])
        self.feat_matrix = normalize(bow_matrix, norm="l2")
        self.item_idx    = {raw_id: idx for idx, raw_id in enumerate(anime_sub.index)}

        self.user_items = {
            u: [(self.item_idx[trainset.to_raw_iid(j)], r)
                for j, r in trainset.ur[u]
                if trainset.to_raw_iid(j) in self.item_idx]
            for u in trainset.all_users()
        }
        return self

    def test(self, testset, verbose=False):
        from collections import defaultdict
        from surprise import Prediction

        user_tests = defaultdict(list)
        for uid, iid, r_ui in testset:
            user_tests[uid].append((iid, r_ui))

        predictions = []
        global_mean = self.trainset.global_mean
        lo, hi = self.trainset.rating_scale

        for raw_uid, test_items in user_tests.items():
            try:
                iuid     = self.trainset.to_inner_uid(raw_uid)
                user_rated = self.user_items.get(iuid, [])
            except ValueError:
                user_rated = []

            known   = [(iid, r) for iid, r in test_items if iid in self.item_idx and user_rated]
            unknown = [(iid, r) for iid, r in test_items if iid not in self.item_idx or not user_rated]

            if known:
                test_idxs  = [self.item_idx[iid] for iid, _ in known]
                rated_idxs = [idx for idx, _ in user_rated]
                ratings    = np.array([r for _, r in user_rated])

                result = self.feat_matrix[test_idxs].dot(self.feat_matrix[rated_idxs].T)
                sims   = result.toarray() if hasattr(result, "toarray") else np.asarray(result)

                for j, (raw_iid, r_ui) in enumerate(known):
                    s   = sims[j]
                    pos = s > 0
                    est = float(s[pos] @ ratings[pos] / s[pos].sum()) if pos.any() else global_mean
                    predictions.append(Prediction(raw_uid, raw_iid, r_ui, min(hi, max(lo, est)), {}))

            for raw_iid, r_ui in unknown:
                predictions.append(Prediction(raw_uid, raw_iid, r_ui, global_mean, {}))

        return predictions

    def estimate(self, u, i):
        if str(i).startswith("UKN__"):
            return self.trainset.global_mean
        raw_i = self.trainset.to_raw_iid(i)
        if raw_i not in self.item_idx or not self.user_items.get(u):
            return self.trainset.global_mean
        vec_i      = np.asarray(self.feat_matrix[self.item_idx[raw_i]].todense()).flatten()
        rated_idxs = [idx for idx, _ in self.user_items[u]]
        ratings    = np.array([r for _, r in self.user_items[u]])
        sims       = np.asarray(self.feat_matrix[rated_idxs].dot(vec_i)).flatten()
        pos        = sims > 0
        if not pos.any():
            return self.trainset.global_mean
        return float(sims[pos] @ ratings[pos] / sims[pos].sum())

In [18]:
bow_rec = BoWRecommender()

bow_rec.fit(trainset)
acc = eval_accuracy(bow_rec.test(testset))

bow_rec.fit(trainset_cov)
cov_preds = bow_rec.test(antitestset_cov)
cov = eval_coverage(cov_preds, catalog, anime_meta)
del cov_preds; gc.collect()

pd.Series({**acc, **cov})

RMSE                      1.4631
MAE                       1.1058
Precision@10              0.7664
Recall@10                 0.5019
NDCG@10                   0.9591
Coverage                 12.7500
Personalization           0.9627
Intra-List Similarity     0.2227
dtype: float64

In [19]:
cv_results = cross_validate(
    BoWRecommender(), data,
    measures=["RMSE", "MAE"],
    cv=KFold(n_splits=5, random_state=SEED),
    return_train_measures=True,
    n_jobs=1,
    verbose=False
)
pd.DataFrame(cv_results)[["test_rmse", "test_mae", "train_rmse", "train_mae"]].agg(["mean", "std"])

,test_rmse,test_mae,train_rmse,train_mae
mean,1.464582,1.106079,1.29110,0.969896
std,0.002306,0.001406,0.00061,0.000429
